# 03 - Gemini: descriptions, model choice, settings, and the test run

**This notebook spends money.** Your Google prepaid credits are the spending limit (the in-code cap is
$1000, effectively off); if they run out, the cell says so - add credit and rerun. Every paid cell prints an estimate *before* spending and the
total spent afterwards. Rough costs (estimates - the spend guard is what really protects the cap): category
descriptions ~$0.3-0.4, tuning check ~$0.4-0.6, test run ~$1.5-2.5. Everything is cached on Drive - rerunning a cell never pays twice for the same work.

**Already picked a model, batch size and thinking level on an earlier run?** Skip Parts A and B: your
saved picks are kept and used automatically.

If a cell says the **daily quota** is used up, rerun that cell tomorrow. If it says the **budget** is reached,
stop: the cap did its job. If something looks wrong, run the **Troubleshoot** cell near the end (free).

**Step 1 - setup (free).** Downloads the project code from GitHub, installs it, connects Google Drive
(all results are saved in `MyDrive/cfpb-nlp-classifier/phase1`), reads your `GEMINI_API_KEY` secret (and the optional `TYPESAFE_API_KEY` Jev secret) and
logs in to Google Sheets. Colab will ask you to allow Drive and Sheets access - click Allow.
Takes 2-4 minutes the first time. If it says the key is missing: click the key icon on the left of
Colab, add a secret named `GEMINI_API_KEY`, and switch "Notebook access" on.
On the first run after a project update, setup may say it moved old results into an `archive_...`
folder on Drive. That is expected and needs nothing from you: your spending record, saved Gemini
answers and saved choices stay where they are.

In [ ]:
!git clone -q https://github.com/Dentoneer/cfpb-nlp-classifier /content/cfpb-nlp-classifier || git -C /content/cfpb-nlp-classifier pull -q
# Installs this project only; never upgrades Colab's own Google libraries.
%pip install -q -e "/content/cfpb-nlp-classifier[ml]"
import sys; sys.path.insert(0, "/content/cfpb-nlp-classifier")
from nlp.colab import setup
cfg, gc, api_key, jev_key = setup()

**Load data and tools (free).** `tuning_df` holds the 200 tuning complaints; `tune` is a random 50 of
them, used for the choices in Parts A and B. The 800 test complaints are not touched until Part D.

In [ ]:
import dataclasses
from nlp import colab, store, pipeline as P
from nlp.llm.classify import run_classification
from nlp.llm.client import GenAIClient
from nlp.llm.describe import estimate_descriptions
complaints = store.read(cfg, P.STAGE_COMPLAINTS)
taxonomy = store.read(cfg, P.STAGE_TAXONOMY)
splits = store.read(cfg, P.STAGE_SPLITS)
train_df, test_df = P.train_test_frames(complaints, splits)
tuning_df = P.split_rows(complaints, splits, splits["split"] == "tuning")
tune = tuning_df.sample(n=min(50, len(tuning_df)), random_state=cfg.seed)
ledger, gem_cache, judge_cache, limiter = colab.open_tools(cfg)
client = GenAIClient(api_key)

## Part 0 - Category descriptions (about $0.3-0.4, once)

Gemini writes a short description for every category (what it covers, when to use it, and which
similar category to use instead), based on example training complaints. The classifier then reads these
descriptions. One request per product. **Estimate first (free):** the next cell prints the worst-case
cost and stops if it will not fit in what is left of the cap.

In [ ]:
usd = estimate_descriptions(cfg, complaints, splits, taxonomy, cache=gem_cache)
colab.check_fits(ledger, usd, "Descriptions (worst case)")

**Write the descriptions (paid, about $0.3-0.4).** Saved on Drive and shown in the taxonomy Google Sheet
(new `description` column). Rerunning is free, and it never overwrites descriptions you edited in the
Sheet: it tells you to load them with the next cell (or set `DISCARD_SHEET_EDITS = True` to throw
them away). **Look at:** the Sheet link and a few descriptions - do
they describe the category sensibly?

In [ ]:
DISCARD_SHEET_EDITS = False   # True only to throw away edits you made in the Sheet
try:
    descriptions = colab.make_descriptions(cfg, complaints, splits, taxonomy, client, ledger,
                                           gem_cache, limiter)
    colab.publish_descriptions(cfg, gc, taxonomy, descriptions,
                               overwrite_edits=DISCARD_SHEET_EDITS)
finally:
    print(colab.ledger_report(ledger))
descriptions[["label", "description"]].head(10)

**Optional - use your own edits (free).** Only if you changed descriptions in the taxonomy Sheet. Edit
only the `description` column (never the `label` column), then run this cell. It checks that every label
is still there and no description is empty, then saves your wording; later Gemini runs use it. Skip this
cell if you made no edits. **Load edits only before Part C/D:** after the Part D test run, new wording
would make notebook 04's self-consistency reruns use a different prompt than the test run.

In [ ]:
descriptions = colab.load_description_edits(cfg, gc, taxonomy)

## Part A - Model bake-off

**Skip Parts A and B if you already picked on an earlier run - your saved picks are kept.**

Runs the same 50 tuning complaints through three Gemini models and compares accuracy against the CFPB
labels. **Estimate first (free):** the next cell prints an upper-bound cost per model and stops if the
total will not fit in what is left of the cap.

In [ ]:
CANDIDATES = ["gemini-3.8-flash", "gemini-3.5-flash-lite", "gemini-3.1-flash-lite"]
est = {m: colab.estimate_classification(cfg, tune, taxonomy, model=m, cache=gem_cache,
                                        classifier_name=f"bakeoff_{m}") for m in CANDIDATES}
for m, usd in est.items():
    print(f"{m:24s} up to ${usd:.2f}")
colab.check_fits(ledger, sum(est.values()), "Bake-off")

**Run the bake-off (~$0.30 or less).** **Look at:** the `path` column (all three levels correct) and
`usd` (what each model actually cost). Higher accuracy for the same or lower dollars wins. If Google
refuses a model, that row says `skipped` with the reason, nothing is charged for it, and the others still run.

In [ ]:
runs = {m: (lambda m=m: run_classification(tune, cfg, client, ledger, gem_cache, f"bakeoff_{m}",
         model=m, taxonomy=taxonomy, limiter=limiter)) for m in CANDIDATES}
bakeoff = colab.compare_runs(runs, tune, ledger)
print(colab.ledger_report(ledger))
bakeoff.round(3)

**Your pick.** Type the winning model in the next cell (replace `None`, keep the quotes) and run it. The
choice is saved on Drive and every later notebook uses it. On a rerun you can leave it as `None`: the
saved pick is kept. The judge model must be a *different* model than the classifier; if you pick
`gemini-3.1-flash-lite`, also set `JUDGE` to another model. Optionally also set `classifier_model` in
`config/phase1.yaml` in the repo so the choice is in git.

In [ ]:
PICK = None    # e.g. "gemini-3.8-flash"
JUDGE = None   # None keeps the current judge model
cfg = colab.apply_choices(cfg, require_pick=True, classifier_model=PICK, judge_model=JUDGE)

## Part B - Batch size and thinking level

Gemini classifies several complaints per request. More per request is cheaper but can be less accurate.
This compares 10 vs 25 per request on the same 50 complaints with your chosen model (~$0.05).
**Look at:** `path` accuracy and `usd`. Keep 10 unless 25 is just as accurate.

In [ ]:
BATCHES = [10, 25]
colab.check_fits(ledger, sum(colab.estimate_classification(cfg, tune, taxonomy, batch_size=b,
    cache=gem_cache, classifier_name=f"batch{b}") for b in BATCHES), "Batch-size check")
runs = {f"batch{b}": (lambda b=b: run_classification(tune, dataclasses.replace(cfg, batch_size=b),
        client, ledger, gem_cache, f"batch{b}", taxonomy=taxonomy, limiter=limiter)) for b in BATCHES}
print(colab.compare_runs(runs, tune, ledger).round(3))

**Thinking level.** Gemini can "think" before answering; thinking tokens are billed like output, so
less thinking is cheaper. Pick the *lowest* level that keeps accuracy (about $0.05-0.10). **Look at:** `path` and `usd`.
(The `minimal` level is left out: gemini-3.8-flash does not support it.)

In [ ]:
LEVELS = ["low", "medium"]
colab.check_fits(ledger, sum(colab.estimate_classification(cfg, tune, taxonomy, thinking=t,
    cache=gem_cache, classifier_name=f"think_{t}") for t in LEVELS), "Thinking check")
runs = {f"think_{t}": (lambda t=t: run_classification(tune, cfg, client, ledger, gem_cache,
        f"think_{t}", thinking=t, taxonomy=taxonomy, limiter=limiter)) for t in LEVELS}
print(colab.compare_runs(runs, tune, ledger).round(3))

**Save your batch size and thinking choices.** Replace `None` with your choice (for example `10` and
`"low"`) and run. `None` keeps the saved value. They apply to every later notebook.

In [ ]:
BATCH_SIZE = None   # 10 or 25
THINKING = None     # "low" or "medium"
cfg = colab.apply_choices(cfg, batch_size=BATCH_SIZE, thinking=THINKING)

## Part C - Check the new prompt on all 200 tuning complaints (about $0.4-0.6)

Classifies all 200 tuning complaints with the new prompt (with the category descriptions from Part 0)
and compares it with **TF-IDF on the same 200 complaints** (from notebook 02, free). It also prints
the first run's numbers for reference (different complaints, so only a rough guide) and, from what this
check really cost, a **projected cost for the rest of the phase**. **Look at:** the `path` column - is
Gemini now ahead of TF-IDF? Rerunning is free. Never tune on the test set.

In [ ]:
try:
    check = colab.prompt_check(cfg, tuning_df, client, ledger, gem_cache, taxonomy, limiter)
finally:
    print(colab.ledger_report(ledger))

## Troubleshoot (optional, free)

Run this only if something looks wrong; it sends nothing to Gemini except a free model list. It prints
your settings, the library version, facts about your key (never the key itself), whether the key works,
the spend so far and the saved answers per run. Copy its output to Claude if you need help.

In [ ]:
colab.diagnose(cfg, api_key, ledger, globals().get("CANDIDATES"))

## Part D - The paid test run

**Estimate before spending (free).** Cost of every remaining paid step in notebooks 03 and 04 (test
classification, two self-consistency reruns, the Jev validator, the summary judge). "estimate" uses what Part C (the
same prompt) really cost per complaint (x1.5 safety); "bound" is the worst case the spend guard
reserves. The next cell stops if the whole remaining phase will not fit in what is left of the cap. (The
guard in code still refuses any single request that could cross the cap.)

In [ ]:
sc_rows = P.split_rows(complaints, splits, splits["self_consistency"])
est = colab.phase_estimate(cfg, ledger, taxonomy, test_df, sc_rows, cache=gem_cache)
colab.check_fits(ledger, colab.pending_usd(est), "Whole remaining phase",
                 bound_usd=colab.pending_usd(est, column="bound_usd"))

**Classify the 800 test complaints (paid, about $1.5-2.5; Part C printed a projection).** Takes 15-40 minutes. If it stops (daily quota
or budget) the message says what to do; finished rows are saved and a rerun continues where it stopped.
**Look at:** accuracy per level vs CFPB labels (compare with the TF-IDF numbers from notebook 02) and the
spend line.

In [ ]:
try:
    gem = colab.stage_paid(cfg, P.STAGE_PRED_GEMINI, lambda: P.run_gemini(
        cfg, test_df, client, ledger, gem_cache, taxonomy, "gemini", limiter),
        test_df["complaint_id"])
    print({k: round(v, 3) for k, v in colab.accuracy_by_level(gem, test_df).items()})
finally:
    print(colab.ledger_report(ledger))

Done. Next: `04_validators.ipynb`.